In [ ]:
%%sql -r CreateSearchService
USE ROLE LOGISTICS_HOL_ADMIN;
USE DATABASE LOGISTICS_HOL;
USE SCHEMA AGENTS;
USE WAREHOUSE HOL_WH;

CREATE OR REPLACE CORTEX SEARCH SERVICE LOGISTICS_HOL.AGENTS.INCIDENT_SEARCH_SVC
  ON REPORT_TEXT
  ATTRIBUTES INCIDENT_ID, SHIPMENT_ID, CARRIER, INCIDENT_TYPE, REGION
  WAREHOUSE = HOL_WH
  TARGET_LAG = '1 hour'
  COMMENT = 'Hybrid search over delivery incident reports.'
  AS (
    SELECT
      INCIDENT_ID,
      SHIPMENT_ID,
      CARRIER,
      INCIDENT_TYPE,
      REGION,
      TO_VARCHAR(INCIDENT_DATE) AS INCIDENT_DATE,
      REPORT_TEXT
    FROM LOGISTICS_HOL.ANALYTICS.DELIVERY_INCIDENTS
  );

SELECT 'Cortex Search service INCIDENT_SEARCH_SVC created' AS STATUS;

In [ ]:
%%sql -r ShowService
SHOW CORTEX SEARCH SERVICES IN SCHEMA LOGISTICS_HOL.AGENTS;

In [ ]:
%%sql -r WhyDhlLate
SELECT PARSE_JSON(
  SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
    'LOGISTICS_HOL.AGENTS.INCIDENT_SEARCH_SVC',
    '{"query": "Why are DHL shipments arriving late?", "columns": ["INCIDENT_ID","CARRIER","INCIDENT_TYPE","REGION","REPORT_TEXT"], "limit": 4}'
  )
)['results'] AS RESULTS;
     

In [ ]:

%%sql -r MechanicalFailures
-- Semantic search finds relevant reports even without exact keywords
SELECT PARSE_JSON(
  SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
    'LOGISTICS_HOL.AGENTS.INCIDENT_SEARCH_SVC',
    '{"query": "fleet maintenance and vehicle breakdowns causing delays", "columns": ["INCIDENT_ID","CARRIER","INCIDENT_TYPE","REPORT_TEXT"], "limit": 3}'
  )
)['results'] AS RESULTS;

In [ ]:
%%sql -r CustomerComplaints
-- Customers threatening to switch carriers
SELECT PARSE_JSON(
  SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
    'LOGISTICS_HOL.AGENTS.INCIDENT_SEARCH_SVC',
    '{"query": "customers considering switching to a different carrier", "columns": ["INCIDENT_ID","CARRIER","INCIDENT_TYPE","REPORT_TEXT"], "limit": 3}'
  )
)['results'] AS RESULTS;